# Drifting ICNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from typing import Tuple, List

### Hyperparameters

In [ ]:
METHODS = ["kernel", "ot_direct", "icnn"]
TARGET = "four_gaussians"
NUM_ITERS = 3000
BATCH_SIZE = 512
LR = 3e-4
INNER_STEPS = 10
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "results/icnn_drifting_comparison_q.png"

# Main ICNN architecture requested for the Gaussian experiments.
ICNN_HIDDEN_DIMS = [512, 256, 128, 64]

# Sinkhorn regularization used by the main ICNN/direct-OT runs.
# The ablation section sweeps this value.
SINKHORN_REG = 0.05

# ICNN initialization: "identity" or "gaussian".
# "gaussian" initializes ∇ψ to the affine OT map between Gaussian approximations
# of the first generated batch and first target batch.
ICNN_INIT_MODE = "gaussian"
ICNN_INIT_BLEND = 1.0  # use 0.5 for a safer but weaker Gaussian start


### ICNN: Input Convex Neural Network

In [ ]:
# ---- ICNN initialization helpers ----

def _symmetrize_matrix(A: torch.Tensor) -> torch.Tensor:
    return 0.5 * (A + A.T)


def psd_matrix_power(A: torch.Tensor, power: float, eps: float = 1e-5) -> torch.Tensor:
    """Stable power of a small symmetric PSD matrix."""
    A = _symmetrize_matrix(A)
    evals, evecs = torch.linalg.eigh(A)
    evals = evals.clamp_min(eps)
    return (evecs * evals.pow(power).unsqueeze(0)) @ evecs.T


def sample_mean_cov(x: torch.Tensor, eps: float = 1e-4) -> Tuple[torch.Tensor, torch.Tensor]:
    """Empirical mean/covariance with diagonal jitter."""
    x = x.detach()
    n, d = x.shape
    mean = x.mean(dim=0)
    xc = x - mean
    denom = max(n - 1, 1)
    cov = (xc.T @ xc) / denom
    cov = cov + eps * torch.eye(d, device=x.device, dtype=x.dtype)
    return mean, cov


def gaussian_ot_affine_map(x_source: torch.Tensor,
                           y_target: torch.Tensor,
                           eps: float = 1e-4) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Closed-form Gaussian OT initialization.

    It returns A, b such that the affine Brenier map
        T0(x) = A x + b
    transports the Gaussian approximation of x_source to the Gaussian
    approximation of y_target.
    """
    m1, S1 = sample_mean_cov(x_source, eps=eps)
    m2, S2 = sample_mean_cov(y_target, eps=eps)

    S1_sqrt = psd_matrix_power(S1, 0.5, eps=eps)
    S1_inv_sqrt = psd_matrix_power(S1, -0.5, eps=eps)
    middle = S1_sqrt @ S2 @ S1_sqrt
    middle_sqrt = psd_matrix_power(middle, 0.5, eps=eps)

    A = S1_inv_sqrt @ middle_sqrt @ S1_inv_sqrt
    A = _symmetrize_matrix(A)
    b = m2 - A @ m1
    return A, b


class ICNN(nn.Module):
    """
    ICNN convex potential with an explicit quadratic base:

        ψ(x) = 1/2 ||M x||² + aᵀx + hidden_residual(x)
        T(x) = ∇ψ(x) = MᵀM x + a + ∇hidden_residual(x)

    Why this fixes initialization:
    - identity init: M=I, a=0, hidden residual has zero input-gradient, so T(x)=x.
    - Gaussian init: MᵀM=A, a=b, so T(x)=Ax+b, the closed-form Gaussian OT map.

    The hidden residual remains convex because hidden-to-hidden weights are projected nonnegative
    and activations are convex/nondecreasing. The quadratic term is always convex because MᵀM is PSD.
    """
    def __init__(self, input_dim: int, hidden_dims: List[int] = [128, 128, 64],
                 beta: float = 20.0, strong_convexity: float = 1.0,
                 activation_bias: float = 1.0):
        super().__init__()
        if len(hidden_dims) < 1:
            raise ValueError("hidden_dims must contain at least one hidden layer")
        self.input_dim = input_dim
        self.beta = beta
        self.num_layers = len(hidden_dims)
        self.activation_bias = activation_bias

        # M in 1/2 ||M x||². This is trainable but convexity is preserved for any M.
        init_scale = float(np.sqrt(max(strong_convexity, 1e-8)))
        self.quad_factor = nn.Parameter(init_scale * torch.eye(input_dim))

        # Linear part aᵀx + c. Linear terms do not affect convexity.
        self.input_skip = nn.Linear(input_dim, 1, bias=True)

        # ICNN residual. Wz_* are direct input-to-hidden weights; Wy_* are nonnegative hidden-to-hidden weights.
        self.Wz_0 = nn.Linear(input_dim, hidden_dims[0], bias=True)
        self.Wy = nn.ModuleList()
        self.Wz = nn.ModuleList()
        for i in range(1, self.num_layers):
            self.Wy.append(nn.Linear(hidden_dims[i-1], hidden_dims[i], bias=False))
            self.Wz.append(nn.Linear(input_dim, hidden_dims[i], bias=True))
        self.Wy_out = nn.Linear(hidden_dims[-1], 1, bias=False)
        self.hidden_bias = nn.Parameter(torch.zeros(1))

        self._init_hidden_residual_zero_gradient()
        self.set_identity_map(strong_convexity=strong_convexity)

    def _init_hidden_residual_zero_gradient(self):
        """
        CONDOT-style identity initializer for the residual:
        - use biases in the high-slope region of softplus, not the saturated negative region;
        - propagate constants through nonnegative averaging weights;
        - initialize input-to-hidden weights to zero, so the residual initially has zero gradient wrt x.
        """
        nn.init.zeros_(self.Wz_0.weight)
        nn.init.constant_(self.Wz_0.bias, self.activation_bias)
        for layer in self.Wz:
            nn.init.zeros_(layer.weight)
            nn.init.constant_(layer.bias, self.activation_bias)
        for layer in list(self.Wy) + [self.Wy_out]:
            fan_in = layer.weight.shape[1]
            with torch.no_grad():
                layer.weight.fill_(1.0 / max(1, fan_in))
        nn.init.zeros_(self.hidden_bias)

    def set_identity_map(self, strong_convexity: float = 1.0):
        """Initialize T(x)=∇ψ(x)≈strong_convexity*x. Use strong_convexity=1 for identity."""
        with torch.no_grad():
            scale = float(np.sqrt(max(strong_convexity, 1e-8)))
            self.quad_factor.copy_(scale * torch.eye(self.input_dim, device=self.quad_factor.device,
                                                     dtype=self.quad_factor.dtype))
            nn.init.zeros_(self.input_skip.weight)
            nn.init.zeros_(self.input_skip.bias)

    def set_affine_gradient(self, A: torch.Tensor, b: torch.Tensor, blend: float = 1.0,
                            eps: float = 1e-5):
        """
        Initialize ∇ψ(x) to the affine map T0(x)=A x + b.
        blend<1 interpolates with identity for a safer start:
            A <- (1-blend)I + blend A, b <- blend b.
        """
        A = A.detach().to(self.quad_factor.device, dtype=self.quad_factor.dtype)
        b = b.detach().to(self.quad_factor.device, dtype=self.quad_factor.dtype)
        I = torch.eye(self.input_dim, device=A.device, dtype=A.dtype)
        blend = float(blend)
        A_blend = _symmetrize_matrix((1.0 - blend) * I + blend * A)
        b_blend = blend * b
        M = psd_matrix_power(A_blend, 0.5, eps=eps)
        with torch.no_grad():
            self.quad_factor.copy_(M)
            self.input_skip.weight.copy_(b_blend.view(1, -1))
            self.input_skip.bias.zero_()

    def set_gaussian_map_from_samples(self, x_source: torch.Tensor, y_target: torch.Tensor,
                                      blend: float = 1.0, eps: float = 1e-4):
        """Initialize ∇ψ to the Gaussian OT map between empirical source/target batches."""
        A, b = gaussian_ot_affine_map(x_source, y_target, eps=eps)
        self.set_affine_gradient(A, b, blend=blend, eps=eps)
        return A, b

    def softplus(self, x):
        return nn.functional.softplus(x, beta=self.beta)

    def forward(self, z):
        Mz = z @ self.quad_factor.T
        quad = 0.5 * (Mz ** 2).sum(-1, keepdim=True)
        linear = self.input_skip(z)

        h = self.softplus(self.Wz_0(z))
        for i in range(len(self.Wy)):
            h = self.softplus(self.Wy[i](h) + self.Wz[i](z))
        hidden = self.Wy_out(h) + self.hidden_bias
        return quad + linear + hidden

    @torch.enable_grad()
    def gradient(self, z, create_graph=None):
        """Compute ∇_z ψ(z). Uses @torch.enable_grad so it works even in no_grad context."""
        if create_graph is None:
            create_graph = self.training
        z_in = z.detach().clone().requires_grad_(True)
        psi = self.forward(z_in)
        return torch.autograd.grad(psi.sum(), z_in, create_graph=create_graph)[0]

    def project_weights(self):
        with torch.no_grad():
            for layer in list(self.Wy) + [self.Wy_out]:
                layer.weight.clamp_(min=0.0)


def compute_V_kernel(x_gen: torch.Tensor, y_pos: torch.Tensor,
                     tau_list: Tuple[float, ...] = (0.02, 0.05, 0.2)):
    """
    Compute the drifting field V(x) following Algorithm 2 / Eq. 11 of the paper.

    Returns V (detached), such that the training loss is:
        target = sg(x_gen + V)
        loss = ||x_gen - target||²

    The loss value equals ||V||², which DECREASES as q → p.
    """
    N, D = x_gen.shape
    M = y_pos.shape[0]
    x = x_gen.detach()
    y = y_pos.detach()

    # Targets: [negatives (=generated), positives (=data)]
    targets = torch.cat([x, y], dim=0)  # [N+M, D]

    # ── Pairwise ℓ₂ distances ──
    dist = torch.cdist(x, targets)  # [N, N+M]

    # ── Distance normalization (Appendix A.6) ──
    # Normalize so mean pairwise distance ≈ √D
    scale = dist.mean().clamp(min=1e-6)
    dist_normed = dist * (np.sqrt(D) / scale)

    # ── Self-masking ──
    diag_mask = torch.zeros(N, N + M, device=x.device)
    diag_mask[:, :N] = torch.eye(N, device=x.device)
    dist_normed = dist_normed + diag_mask * 100.0

    # ── Force accumulation (NO per-temperature normalization) ──
    V = torch.zeros_like(x)

    for tau in tau_list:
        logits = -dist_normed / tau

        # Double softmax (paper's Alg. 2: softmax over y, then over x)
        aff_row = torch.softmax(logits, dim=-1)
        aff_col = torch.softmax(logits, dim=-2)
        affinity = torch.sqrt((aff_row * aff_col).clamp(min=1e-6))

        aff_neg = affinity[:, :N]      # negative affinities
        aff_pos = affinity[:, N:]      # positive affinities
        sum_pos = aff_pos.sum(-1, keepdim=True)
        sum_neg = aff_neg.sum(-1, keepdim=True)

        # Drifting coefficients: attract by positives, repel by negatives
        r_coeff_neg = -aff_neg * sum_pos
        r_coeff_pos = aff_pos * sum_neg
        R_coeff = torch.cat([r_coeff_neg, r_coeff_pos], dim=1)  # [N, N+M]

        # Force = weighted combination of (target - x) vectors
        force_R = R_coeff @ targets - R_coeff.sum(-1, keepdim=True) * x
        V = V + force_R  # Raw force, NO normalization

    return V.detach()

### ICNN-based V (Sinkhorn OT displacement)

In [ ]:
def sinkhorn(C, reg=0.05, num_iters=100):
    """
    Simple Sinkhorn solver used for the toy 2D experiments.

    Note: this is fine for the current Gaussian proof-of-concept. If you increase
    the dimension or use very small reg, switch to a log-domain Sinkhorn solver.
    """
    K = torch.exp(-C / (reg + 1e-8))
    u = torch.ones(C.shape[0], device=C.device)
    v = torch.ones(C.shape[1], device=C.device)
    for _ in range(num_iters):
        u = 1.0 / (K @ v + 1e-8)
        v = 1.0 / (K.T @ u + 1e-8)
    return u.unsqueeze(1) * K * v.unsqueeze(0)


def compute_V_ot_direct(x_gen: torch.Tensor, y_pos: torch.Tensor,
                        reg: float = 0.05):
    """
    Direct OT displacement V(x) = T_Sinkhorn(x) - x.
    No ICNN inner loop: uses Sinkhorn soft assignment directly.
    """
    x = x_gen.detach()
    y = y_pos.detach()
    C = torch.cdist(x, y, p=2) ** 2
    P = sinkhorn(C, reg=reg)
    P_row = P / (P.sum(1, keepdim=True) + 1e-8)
    y_target = P_row @ y
    return (y_target - x).detach()


class ICNNDriftField:
    """V(x) = ∇ψ_ω(x) - x, with identity or Gaussian OT initialization."""
    def __init__(self, dim, hidden_dims=None,
                 inner_steps=5, inner_lr=1e-2,
                 sinkhorn_reg=0.05, strong_convexity=1.0,
                 init_mode="identity", init_blend=1.0, init_eps=1e-4):
        if hidden_dims is None:
            hidden_dims = ICNN_HIDDEN_DIMS
        self.hidden_dims = list(hidden_dims)
        self.icnn = ICNN(dim, self.hidden_dims, strong_convexity=strong_convexity)
        self.optimizer = optim.Adam(self.icnn.parameters(), lr=inner_lr)
        self.inner_steps = inner_steps
        self.sinkhorn_reg = sinkhorn_reg
        self.inner_lr = inner_lr
        self.init_mode = init_mode
        self.init_blend = init_blend
        self.init_eps = init_eps
        self._did_gaussian_init = False

    def to(self, device):
        self.icnn = self.icnn.to(device)
        self.optimizer = optim.Adam(self.icnn.parameters(), lr=self.inner_lr)
        return self

    def _maybe_gaussian_initialize(self, x, y):
        if self.init_mode == "gaussian" and not self._did_gaussian_init:
            A, b = self.icnn.set_gaussian_map_from_samples(
                x, y, blend=self.init_blend, eps=self.init_eps
            )
            # Reset Adam state after manually changing parameters.
            self.optimizer = optim.Adam(self.icnn.parameters(), lr=self.inner_lr)
            self._did_gaussian_init = True
            with torch.no_grad():
                T0 = x @ A.T + b
                init_v = ((T0 - x) ** 2).mean().item()
                print(f"  [ICNN gaussian init] hidden_dims={self.hidden_dims} | initial affine ||T0(x)-x||² = {init_v:.6f}")

    def compute_V(self, x_gen, y_pos):
        """
        V(x) = T_q→p(x) - x = ∇ψ(x) - x
        where ψ is an ICNN trained to approximate the OT map from q to p.
        """
        x = x_gen.detach()
        y = y_pos.detach()

        # Optional Gaussian OT initialization from the first batch.
        # This is done once, not every iteration, so warm-starting is preserved.
        self._maybe_gaussian_initialize(x, y)

        # Step 1: Sinkhorn assignment to get OT target for each x.
        with torch.no_grad():
            C = torch.cdist(x, y, p=2) ** 2
            P = sinkhorn(C, reg=self.sinkhorn_reg)
            P_row = P / (P.sum(1, keepdim=True) + 1e-8)
            y_target = P_row @ y  # Barycentric OT target

        # Step 2: Train ICNN to fit ∇ψ(x) ≈ y_target.
        self.icnn.train()
        for _ in range(self.inner_steps):
            self.optimizer.zero_grad()
            T_x = self.icnn.gradient(x)  # create_graph=True while training
            loss = ((T_x - y_target) ** 2).mean()
            loss.backward()
            self.optimizer.step()
            self.icnn.project_weights()

        # Step 3: Return displacement V = ∇ψ(x) - x.
        self.icnn.eval()
        T_x = self.icnn.gradient(x, create_graph=False)
        return (T_x.detach() - x).detach()


### Generator

In [ ]:
class ToyGenerator(nn.Module):
    """
    Small MLP: ε ∈ R² → x ∈ R².

    The initial output distribution must span the target range.
    If the initial generator output is concentrated (std << target range),
    the kernel V gives near-zero force (all data points equidistant).
    """
    def __init__(self, noise_dim=2, data_dim=2, hidden_dim=256, output_scale=2.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
        # Initialize so output has std ≈ output_scale
        # We need to account for variance propagation through the MLP
        self._init_output_scale(output_scale)

    def _init_output_scale(self, target_std):
        """Initialize so the output std ≈ target_std."""
        with torch.no_grad():
            # Check current output scale
            test_input = torch.randn(1000, 2)
            test_output = self.net(test_input)
            current_std = test_output.std().item()
            if current_std > 1e-6:
                scale_factor = target_std / current_std
                # Scale only the last layer to avoid disrupting internal representations
                self.net[-1].weight.mul_(scale_factor)
                self.net[-1].bias.mul_(scale_factor)

    def forward(self, eps):
        return self.net(eps)

### Target Distributions

In [ ]:
def sample_bimodal(n, device):
    mix = torch.rand(n, device=device) < 0.5
    x = torch.randn(n, 2, device=device) * 0.4
    x[mix, 0] += 2.0
    x[~mix, 0] -= 2.0
    return x

def sample_ring(n, device):
    theta = 2 * np.pi * torch.rand(n, device=device)
    r = 2.0 + 0.2 * torch.randn(n, device=device)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], -1)

def sample_four_gaussians(n, device):
    c = torch.tensor([[-2, -2], [-2, 2], [2, -2], [2, 2]],
                      device=device, dtype=torch.float32)
    return c[torch.randint(0, 4, (n,), device=device)] + 0.3 * torch.randn(n, 2, device=device)

### Training Loop

In [ ]:
def train(method='kernel', target='bimodal', num_iters=3000, batch_size=512,
          lr=3e-4, inner_steps=10, seed=42,
          sinkhorn_reg=SINKHORN_REG, icnn_hidden_dims=None,
          device='cuda' if torch.cuda.is_available() else 'cpu'):

    torch.manual_seed(seed)
    device = torch.device(device)
    samplers = dict(bimodal=sample_bimodal, ring=sample_ring,
                    four_gaussians=sample_four_gaussians)
    sample_target = samplers[target]

    if icnn_hidden_dims is None:
        icnn_hidden_dims = ICNN_HIDDEN_DIMS

    gen = ToyGenerator(hidden_dim=256, output_scale=2.0).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr)

    with torch.no_grad():
        test_out = gen(torch.randn(1000, 2, device=device))
        print(f"  [{method}] Initial output: mean={test_out.mean(0).cpu().numpy()}, "
              f"std={test_out.std(0).cpu().numpy()}")

    icnn_drift = None
    if method == 'icnn':
        icnn_drift = ICNNDriftField(
            dim=2, hidden_dims=icnn_hidden_dims,
            inner_steps=inner_steps, inner_lr=1e-2,
            sinkhorn_reg=sinkhorn_reg, strong_convexity=1.0,
            init_mode=ICNN_INIT_MODE, init_blend=ICNN_INIT_BLEND,
        ).to(device)
        with torch.no_grad():
            tz = torch.randn(8, 2, device=device)
            err = (icnn_drift.icnn.gradient(tz, False) - tz).abs().max().item()
            print(f"  [pre-batch init check] hidden_dims={icnn_hidden_dims} | max |T(z)-z| = {err:.2e}; init_mode={ICNN_INIT_MODE}")

    losses, v_norms, snapshots = [], [], []

    for it in range(num_iters):
        eps = torch.randn(batch_size, 2, device=device)
        y_pos = sample_target(batch_size, device)
        x_gen = gen(eps)

        if method == 'kernel':
            V = compute_V_kernel(x_gen, y_pos, tau_list=(0.02, 0.05, 0.2))
        elif method == 'ot_direct':
            V = compute_V_ot_direct(x_gen, y_pos, reg=sinkhorn_reg)
        else:
            V = icnn_drift.compute_V(x_gen, y_pos)

        # Drifting loss:
        # L = ||f(ε) - sg(f(ε) + V(f(ε)))||²
        target_pts = (x_gen.detach() + V).detach()  # frozen target
        loss = ((x_gen - target_pts) ** 2).mean()
        v_norm = (V ** 2).mean().item()  # ||V||² — should decrease

        gen_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        losses.append(loss.item())
        v_norms.append(v_norm)

        if it % (num_iters // 8) == 0 or it == num_iters - 1:
            with torch.no_grad():
                snap_x = gen(torch.randn(1000, 2, device=device)).cpu().numpy()
            snapshots.append((it, snap_x))
            print(f"[{method}] iter {it:5d} | loss: {loss.item():.6f} | "
                  f"||V||²: {v_norm:.6f}")

    return dict(losses=losses, v_norms=v_norms, snapshots=snapshots,
                method=method, target=target,
                sinkhorn_reg=float(sinkhorn_reg),
                icnn_hidden_dims=list(icnn_hidden_dims) if method == 'icnn' else None)


In [ ]:
results = []
for m in METHODS:
    print(f"\n{'=' * 60}\nTraining: {m}\n{'=' * 60}")
    results.append(train(
        method=m, target=TARGET, num_iters=NUM_ITERS,
        batch_size=BATCH_SIZE, lr=LR,
        inner_steps=INNER_STEPS, seed=SEED,
        sinkhorn_reg=SINKHORN_REG,
        icnn_hidden_dims=ICNN_HIDDEN_DIMS,
        device=DEVICE))


### Visualization

In [ ]:
def plot_results(results_list, target='bimodal', device='cpu',
                 save=False, save_path='icnn_drifting_comparison.png'):
    n_methods = len(results_list)
    n_snaps = len(results_list[0]['snapshots'])
    fig = plt.figure(figsize=(4 * n_snaps + 4, 4 * n_methods + 4))
    gs = GridSpec(n_methods + 1, n_snaps + 1, figure=fig, hspace=0.3, wspace=0.3)

    samplers = dict(bimodal=sample_bimodal, ring=sample_ring,
                    four_gaussians=sample_four_gaussians)
    y_gt = samplers[target](2000, torch.device('cpu')).numpy()

    for row, res in enumerate(results_list):
        for col, (it, snap) in enumerate(res['snapshots']):
            ax = fig.add_subplot(gs[row, col])
            ax.scatter(y_gt[:, 0], y_gt[:, 1], s=1, alpha=.3, c='blue', label='target $p$')
            ax.scatter(snap[:, 0], snap[:, 1], s=1, alpha=.3, c='orange', label='generated $q$')
            ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
            ax.set_title(f'iter {it}', fontsize=10)
            if col == 0:
                ax.set_ylabel(res['method'], fontsize=12, fontweight='bold')
            if row == 0 and col == 0:
                ax.legend(fontsize=7, loc='upper left')
        ax_l = fig.add_subplot(gs[row, -1])
        ax_l.semilogy(res['v_norms'], lw=.8)
        ax_l.set_xlabel('iter'); ax_l.set_ylabel('$\\|V\\|^2$'); ax_l.grid(True, alpha=.3)
        ax_l.set_title(f'{res["method"]}: $\\|V\\|^2$', fontsize=10)

    ax = fig.add_subplot(gs[-1, :])
    for i, res in enumerate(results_list):
        ax.semilogy(res['v_norms'], label=res['method'], lw=1.5, alpha=.8)
    ax.set_xlabel('iteration', fontsize=12)
    ax.set_ylabel('$\\|V\\|^2$ (log)', fontsize=12)
    ax.set_title('Convergence Comparison', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11); ax.grid(True, alpha=.3)

    if save:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")

    plt.close(fig)
    return fig

In [ ]:
plot_results(results, TARGET, DEVICE)

## Gaussian metrics

Run the helper below after training to turn the current visual result into preliminary quantitative evidence.

In [ ]:

# Suggested metrics for four-Gaussian preliminary results.
# Run after `results = [...]` has been computed.

def four_gaussian_metrics(samples, radius=0.75):
    """Simple held-out diagnostics for the four-Gaussian target."""
    x = torch.as_tensor(samples, dtype=torch.float32)
    centers = torch.tensor([[-2., -2.], [-2., 2.], [2., -2.], [2., 2.]])
    d = torch.cdist(x, centers)
    nearest = d.argmin(dim=1)
    min_dist = d.min(dim=1).values

    counts = torch.bincount(nearest, minlength=4).float()
    mass = counts / counts.sum().clamp_min(1.0)
    close_mass = torch.stack([
        ((nearest == k) & (min_dist < radius)).float().mean()
        for k in range(4)
    ])
    captured = int((close_mass > 0.02).sum().item())
    off_manifold = float((min_dist >= radius).float().mean().item())

    return {
        "captured_modes": captured,
        "mass_per_mode": mass.tolist(),
        "close_mass_per_mode": close_mass.tolist(),
        "off_manifold_frac": off_manifold,
        "mean_nearest_center_dist": float(min_dist.mean().item()),
    }

for res in results:
    final_samples = res["snapshots"][-1][1]
    print(res["method"], four_gaussian_metrics(final_samples))


## ICNN initialization sanity check

This checks the two initializers before running long experiments. With `identity`, `∇ψ(x)` should be almost exactly `x`. With `gaussian`, `∇ψ(x)` should match the closed-form affine Gaussian OT map computed from one generated batch and one target batch.


In [ ]:
def check_icnn_initializations(target=TARGET, batch_size=BATCH_SIZE, device=DEVICE, seed=SEED):
    torch.manual_seed(seed)
    device = torch.device(device)
    sample_target = dict(
        bimodal=sample_bimodal,
        ring=sample_ring,
        four_gaussians=sample_four_gaussians,
    )[target]

    gen = ToyGenerator(hidden_dim=256, output_scale=2.0).to(device)
    x = gen(torch.randn(batch_size, 2, device=device)).detach()
    y = sample_target(batch_size, device).detach()

    icnn_identity = ICNN(2, ICNN_HIDDEN_DIMS).to(device)
    identity_err = (icnn_identity.gradient(x, create_graph=False) - x).abs().max().item()

    icnn_gaussian = ICNN(2, ICNN_HIDDEN_DIMS).to(device)
    A, b = icnn_gaussian.set_gaussian_map_from_samples(x, y, blend=1.0)
    T_affine = x @ A.T + b
    T_icnn = icnn_gaussian.gradient(x, create_graph=False)
    gaussian_err = (T_icnn - T_affine).abs().max().item()

    print(f"hidden dims: {ICNN_HIDDEN_DIMS}")
    print(f"identity init: max |∇ψ(x)-x| = {identity_err:.3e}")
    print(f"gaussian init: max |∇ψ(x)-(Ax+b)| = {gaussian_err:.3e}")
    print(f"gaussian init drift norm ||Ax+b-x||² = {((T_affine - x) ** 2).mean().item():.6f}")
    return {
        "identity_err": identity_err,
        "gaussian_err": gaussian_err,
        "gaussian_init_v_norm": ((T_affine - x) ** 2).mean().item(),
        "A": A.detach().cpu().numpy(),
        "b": b.detach().cpu().numpy(),
    }

# Run this once before the ablation.
init_check = check_icnn_initializations()
init_check


## ICNN architecture + Sinkhorn regularization ablation

This section answers two questions:

1. **ICNN expressiveness:** what happens to the drift norm / loss as the ICNN becomes wider and deeper?
2. **Sinkhorn regularization:** how sensitive are the ICNN targets and final drift norm to `sinkhorn_reg`?

The architecture convention is now tapered:

```python
make_tapered_hidden_dims(width=512, depth=4) == [512, 256, 128, 64]
```

So this ablation does **not** test `[512, 512, 512, 512]`. It tests progressively larger tapered ICNNs.

Main quantities to report:

- `tail_mean_v_norm`: moving average of `||V||²` over the last 10% of training.
- `tail_mean_inner_loss`: whether `∇ψ(x)` is actually fitting the Sinkhorn barycentric target.
- `tail_mean_target_v_norm`: how strong the direct Sinkhorn target is for that regularization.
- `final_fit_ratio`: whether the ICNN drift is much smaller/larger than the target drift.
- `time_per_iter_sec`: cost of wider/deeper ICNNs.


In [ ]:
import os, time, math
import pandas as pd


def count_parameters(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def make_tapered_hidden_dims(width=512, depth=4, taper=2, min_width=16):
    """
    Tapered ICNN architecture helper.

    Examples:
        make_tapered_hidden_dims(512, 4) -> [512, 256, 128, 64]
        make_tapered_hidden_dims(256, 3) -> [256, 128, 64]

    This replaces the repeated-width convention [width] * depth.
    """
    dims = []
    for i in range(int(depth)):
        dims.append(max(int(min_width), int(round(width / (taper ** i)))))
    return dims


class ICNNDriftFieldWithStats:
    """
    Same as ICNNDriftField, but returns diagnostics for ablations.

    `init_mode="gaussian"` initializes the quadratic base of ψ so that ∇ψ is the
    closed-form Gaussian OT affine map between the first generated batch and the
    first target batch.
    """
    def __init__(self, dim, hidden_dims,
                 inner_steps=10, inner_lr=1e-2,
                 sinkhorn_reg=0.05, strong_convexity=1.0,
                 init_mode="identity", init_blend=1.0, init_eps=1e-4):
        self.hidden_dims = list(hidden_dims)
        self.icnn = ICNN(dim, self.hidden_dims, strong_convexity=strong_convexity)
        self.optimizer = optim.Adam(self.icnn.parameters(), lr=inner_lr)
        self.inner_steps = inner_steps
        self.sinkhorn_reg = sinkhorn_reg
        self.inner_lr = inner_lr
        self.init_mode = init_mode
        self.init_blend = init_blend
        self.init_eps = init_eps
        self._did_gaussian_init = False

    def to(self, device):
        self.icnn = self.icnn.to(device)
        self.optimizer = optim.Adam(self.icnn.parameters(), lr=self.inner_lr)
        return self

    def _maybe_gaussian_initialize(self, x, y):
        if self.init_mode == "gaussian" and not self._did_gaussian_init:
            A, b = self.icnn.set_gaussian_map_from_samples(
                x, y, blend=self.init_blend, eps=self.init_eps
            )
            self.optimizer = optim.Adam(self.icnn.parameters(), lr=self.inner_lr)
            self._did_gaussian_init = True
            with torch.no_grad():
                T0 = x @ A.T + b
                init_v = ((T0 - x) ** 2).mean().item()
            return init_v
        return None

    def compute_V(self, x_gen, y_pos):
        """
        Returns:
            V: detached ICNN drift, shape [batch, 2]
            stats: dict with inner_loss, target_v_norm, fit_ratio, optional gaussian_init_v_norm
        """
        x = x_gen.detach()
        y = y_pos.detach()

        gaussian_init_v_norm = self._maybe_gaussian_initialize(x, y)

        # Sinkhorn barycentric target y_bar.
        with torch.no_grad():
            C = torch.cdist(x, y, p=2) ** 2
            P = sinkhorn(C, reg=self.sinkhorn_reg)
            P_row = P / (P.sum(1, keepdim=True) + 1e-8)
            y_target = P_row @ y
            target_v_norm = ((y_target - x) ** 2).mean().item()

        # Inner-loop ICNN regression: fit ∇ψ(x) to y_target.
        self.icnn.train()
        last_inner_loss = None
        for _ in range(self.inner_steps):
            self.optimizer.zero_grad()
            T_x = self.icnn.gradient(x)  # create_graph=True while training
            inner_loss = ((T_x - y_target) ** 2).mean()
            inner_loss.backward()
            self.optimizer.step()
            self.icnn.project_weights()
            last_inner_loss = inner_loss.item()

        # Drift returned to the outer generator update.
        self.icnn.eval()
        T_x = self.icnn.gradient(x, create_graph=False)
        V = (T_x.detach() - x).detach()
        v_norm = (V ** 2).mean().item()

        stats = {
            "inner_loss": float(last_inner_loss),
            "target_v_norm": float(target_v_norm),
            "fit_ratio": float(v_norm / (target_v_norm + 1e-8)),
            "gaussian_init_v_norm": None if gaussian_init_v_norm is None else float(gaussian_init_v_norm),
        }
        return V, stats


def train_icnn_configuration(
    hidden_dims=None,
    width=None,
    depth=None,
    target="four_gaussians",
    num_iters=1000,
    batch_size=512,
    lr=3e-4,
    inner_steps=10,
    seed=42,
    device=None,
    sinkhorn_reg=0.05,
    inner_lr=1e-2,
    strong_convexity=1.0,
    init_mode="identity",
    init_blend=1.0,
    log_every=None,
):
    """
    Train only the ICNN method for one architecture + one Sinkhorn regularization.

    You can either pass hidden_dims directly or pass (width, depth). If width/depth
    are used, hidden dims are tapered:
        hidden_dims = make_tapered_hidden_dims(width, depth)
    """
    if device is None:
        device = DEVICE
    device = torch.device(device)

    if hidden_dims is None:
        if width is None or depth is None:
            hidden_dims = ICNN_HIDDEN_DIMS
        else:
            hidden_dims = make_tapered_hidden_dims(width=width, depth=depth)
    hidden_dims = list(map(int, hidden_dims))
    width = int(hidden_dims[0])
    depth = int(len(hidden_dims))
    hidden_dims_str = "-".join(map(str, hidden_dims))

    # Same generator initialization for all configs with the same seed.
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    samplers = dict(
        bimodal=sample_bimodal,
        ring=sample_ring,
        four_gaussians=sample_four_gaussians,
    )
    sample_target = samplers[target]

    gen = ToyGenerator(hidden_dim=256, output_scale=2.0).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr)

    icnn_drift = ICNNDriftFieldWithStats(
        dim=2,
        hidden_dims=hidden_dims,
        inner_steps=inner_steps,
        inner_lr=inner_lr,
        sinkhorn_reg=sinkhorn_reg,
        strong_convexity=strong_convexity,
        init_mode=init_mode, init_blend=init_blend,
    ).to(device)

    # Reset RNG after architecture-dependent ICNN initialization.
    # This makes the noise/data stream more comparable across widths/depths/regs.
    torch.manual_seed(seed + 10_000)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed + 10_000)

    v_norms, losses, inner_losses, target_v_norms, fit_ratios = [], [], [], [], []
    gaussian_init_v_norm = None

    if log_every is None:
        log_every = max(1, num_iters // 5)

    t0 = time.time()
    for it in range(num_iters):
        eps = torch.randn(batch_size, 2, device=device)
        y_pos = sample_target(batch_size, device)
        x_gen = gen(eps)

        V, stats = icnn_drift.compute_V(x_gen, y_pos)

        target_pts = (x_gen.detach() + V).detach()
        loss = ((x_gen - target_pts) ** 2).mean()

        gen_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        v_norm = (V ** 2).mean().item()
        losses.append(loss.item())
        v_norms.append(v_norm)
        inner_losses.append(stats["inner_loss"])
        target_v_norms.append(stats["target_v_norm"])
        fit_ratios.append(stats["fit_ratio"])
        if stats.get("gaussian_init_v_norm") is not None:
            gaussian_init_v_norm = stats["gaussian_init_v_norm"]

        if it % log_every == 0 or it == num_iters - 1:
            print(
                f"[dims={hidden_dims_str}, reg={sinkhorn_reg:g}, seed={seed}] "
                f"iter {it:5d}/{num_iters} | "
                f"||V||²={v_norm:.6f} | "
                f"inner={stats['inner_loss']:.6f} | "
                f"target={stats['target_v_norm']:.6f}"
            )

    elapsed = time.time() - t0
    tail = max(1, num_iters // 10)

    with torch.no_grad():
        final_samples = gen(torch.randn(2000, 2, device=device)).cpu().numpy()

    return {
        "width": int(width),
        "depth": int(depth),
        "hidden_dims": hidden_dims,
        "hidden_dims_str": hidden_dims_str,
        "seed": int(seed),
        "target": target,
        "num_iters": int(num_iters),
        "batch_size": int(batch_size),
        "inner_steps": int(inner_steps),
        "sinkhorn_reg": float(sinkhorn_reg),
        "inner_lr": float(inner_lr),
        "strong_convexity": float(strong_convexity),
        "init_mode": init_mode,
        "init_blend": float(init_blend),
        "gaussian_init_v_norm": gaussian_init_v_norm,
        "param_count": int(count_parameters(icnn_drift.icnn)),
        "time_sec": float(elapsed),
        "time_per_iter_sec": float(elapsed / max(1, num_iters)),
        "losses": losses,
        "v_norms": v_norms,
        "inner_losses": inner_losses,
        "target_v_norms": target_v_norms,
        "fit_ratios": fit_ratios,
        "final_samples": final_samples,
        "final_v_norm": float(v_norms[-1]),
        "tail_mean_v_norm": float(np.mean(v_norms[-tail:])),
        "best_v_norm": float(np.min(v_norms)),
        "final_inner_loss": float(inner_losses[-1]),
        "tail_mean_inner_loss": float(np.mean(inner_losses[-tail:])),
        "final_target_v_norm": float(target_v_norms[-1]),
        "tail_mean_target_v_norm": float(np.mean(target_v_norms[-tail:])),
        "final_fit_ratio": float(fit_ratios[-1]),
    }


# Backwards-compatible alias in case you already used the previous ablation cell name.
def train_icnn_architecture(width=128, depth=3, **kwargs):
    return train_icnn_configuration(width=width, depth=depth, **kwargs)


def run_icnn_architecture_sinkhorn_ablation(
    widths=(128, 256, 512),
    depths=(2, 3, 4),
    sinkhorn_regs=(0.02, 0.05, 0.1),
    seeds=(42,),
    target="four_gaussians",
    num_iters=1000,
    batch_size=512,
    inner_steps=10,
    lr=3e-4,
    inner_lr=1e-2,
    strong_convexity=1.0,
    init_mode="identity",
    init_blend=1.0,
    device=None,
    save_dir="results/icnn_architecture_sinkhorn_ablation",
):
    """
    Runs a joint ablation over tapered ICNN architecture and Sinkhorn regularization.

    Quick first pass:
        widths=(256, 512), depths=(3, 4), sinkhorn_regs=(0.02, 0.05, 0.1),
        seeds=(42,), num_iters=600

    Report-quality:
        widths=(128, 256, 512), depths=(2, 3, 4, 5),
        sinkhorn_regs=(0.01, 0.02, 0.05, 0.1, 0.2),
        seeds=(0, 1, 2, 3, 4), num_iters=3000
    """
    os.makedirs(save_dir, exist_ok=True)
    records, runs = [], []

    total = len(widths) * len(depths) * len(sinkhorn_regs) * len(seeds)
    job = 0

    for reg in sinkhorn_regs:
        for depth in depths:
            for width in widths:
                hidden_dims = make_tapered_hidden_dims(width=width, depth=depth)
                for seed in seeds:
                    job += 1
                    print("\n" + "=" * 90)
                    print(f"Ablation job {job}/{total}: hidden_dims={hidden_dims}, sinkhorn_reg={reg}, seed={seed}")
                    print("=" * 90)

                    run = train_icnn_configuration(
                        hidden_dims=hidden_dims,
                        target=target,
                        num_iters=num_iters,
                        batch_size=batch_size,
                        lr=lr,
                        inner_steps=inner_steps,
                        seed=seed,
                        device=device,
                        sinkhorn_reg=reg,
                        inner_lr=inner_lr,
                        strong_convexity=strong_convexity,
                        init_mode=init_mode,
                        init_blend=init_blend,
                    )

                    runs.append(run)
                    records.append({
                        "width": run["width"],
                        "depth": run["depth"],
                        "hidden_dims_str": run["hidden_dims_str"],
                        "seed": run["seed"],
                        "sinkhorn_reg": run["sinkhorn_reg"],
                        "param_count": run["param_count"],
                        "final_v_norm": run["final_v_norm"],
                        "tail_mean_v_norm": run["tail_mean_v_norm"],
                        "best_v_norm": run["best_v_norm"],
                        "final_inner_loss": run["final_inner_loss"],
                        "tail_mean_inner_loss": run["tail_mean_inner_loss"],
                        "final_target_v_norm": run["final_target_v_norm"],
                        "tail_mean_target_v_norm": run["tail_mean_target_v_norm"],
                        "final_fit_ratio": run["final_fit_ratio"],
                        "time_sec": run["time_sec"],
                        "time_per_iter_sec": run["time_per_iter_sec"],
                        "num_iters": run["num_iters"],
                        "batch_size": run["batch_size"],
                        "inner_steps": run["inner_steps"],
                        "inner_lr": run["inner_lr"],
                        "strong_convexity": run["strong_convexity"],
                        "init_mode": run["init_mode"],
                        "init_blend": run["init_blend"],
                        "gaussian_init_v_norm": run["gaussian_init_v_norm"],
                    })

                    df = pd.DataFrame(records)
                    df.to_csv(os.path.join(save_dir, "ablation_summary_partial.csv"), index=False)

    df = pd.DataFrame(records)
    df.to_csv(os.path.join(save_dir, "ablation_summary.csv"), index=False)
    return df, runs


# Backwards-compatible alias from the previous notebook version.
def run_width_depth_ablation(widths=(128, 256, 512), depths=(2, 3, 4), seeds=(42,), **kwargs):
    return run_icnn_architecture_sinkhorn_ablation(
        widths=widths, depths=depths, seeds=seeds,
        sinkhorn_regs=(kwargs.pop("sinkhorn_reg", SINKHORN_REG),),
        **kwargs,
    )


def summarize_ablation(df):
    """
    Aggregates across seeds.
    Use `tail_mean_v_norm_mean` as your main drift-norm number.
    """
    metrics = [
        "param_count",
        "final_v_norm",
        "tail_mean_v_norm",
        "best_v_norm",
        "final_inner_loss",
        "tail_mean_inner_loss",
        "final_target_v_norm",
        "tail_mean_target_v_norm",
        "final_fit_ratio",
        "time_per_iter_sec",
        "gaussian_init_v_norm",
    ]

    grouped = (
        df.groupby(["sinkhorn_reg", "depth", "width", "hidden_dims_str"])[metrics]
        .agg(["mean", "std"])
        .reset_index()
    )

    # Flatten column names.
    grouped.columns = [
        "_".join([str(x) for x in col if x != ""]).strip("_")
        for col in grouped.columns.to_flat_index()
    ]

    grouped = grouped.sort_values("tail_mean_v_norm_mean")
    return grouped


def plot_architecture_heatmap(df, metric="tail_mean_v_norm", sinkhorn_reg=None, save_path=None):
    """
    Heatmap: rows = depth, columns = width.
    Cell values are averaged over seeds.

    If sinkhorn_reg is provided, the heatmap is for one regularization value.
    """
    if sinkhorn_reg is not None:
        plot_df = df[np.isclose(df["sinkhorn_reg"], sinkhorn_reg)].copy()
        title_suffix = f" | sinkhorn_reg={sinkhorn_reg:g}"
    else:
        plot_df = df.copy()
        title_suffix = " | averaged over sinkhorn_reg"

    pivot = plot_df.groupby(["depth", "width"])[metric].mean().unstack("width")
    depths = list(pivot.index)
    widths = list(pivot.columns)

    fig, ax = plt.subplots(figsize=(1.4 * len(widths) + 3, 1.0 * len(depths) + 2))
    im = ax.imshow(pivot.values, aspect="auto")

    ax.set_xticks(range(len(widths)))
    ax.set_xticklabels(widths)
    ax.set_yticks(range(len(depths)))
    ax.set_yticklabels(depths)
    ax.set_xlabel("ICNN starting width, tapered by /2 each layer")
    ax.set_ylabel("ICNN depth")
    ax.set_title(metric + title_suffix)

    for i in range(len(depths)):
        for j in range(len(widths)):
            val = pivot.values[i, j]
            ax.text(j, i, f"{val:.3g}", ha="center", va="center")

    fig.colorbar(im, ax=ax, label=metric)
    fig.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()
    return fig, ax


def plot_sinkhorn_reg_ablation(df, metric="tail_mean_v_norm", save_path=None):
    """
    Plot metric vs sinkhorn_reg for each tapered ICNN architecture.
    Values are averaged over seeds.
    """
    mean_df = (
        df.groupby(["hidden_dims_str", "sinkhorn_reg"])[metric]
        .mean()
        .reset_index()
        .sort_values("sinkhorn_reg")
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    for hidden_dims_str, sub in mean_df.groupby("hidden_dims_str"):
        ax.plot(sub["sinkhorn_reg"], sub[metric], marker="o", label=hidden_dims_str)

    ax.set_xscale("log")
    ax.set_xlabel("Sinkhorn regularization")
    ax.set_ylabel(metric)
    ax.set_title(f"Sinkhorn regularization ablation: {metric}")
    ax.grid(True, alpha=0.3)
    if mean_df["hidden_dims_str"].nunique() <= 10:
        ax.legend(title="hidden dims", fontsize=8)
    fig.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()
    return fig, ax


def plot_ablation_curves(runs, metric="v_norms", save_path=None, max_curves=15):
    """
    Plot curves. For readability, only plot the first `max_curves` runs by default.
    """
    fig, ax = plt.subplots(figsize=(8, 5))

    for run in runs[:max_curves]:
        label = f"dims={run['hidden_dims_str']}, reg={run['sinkhorn_reg']:g}, seed={run['seed']}"
        y = run[metric]
        ax.semilogy(y, linewidth=1.0, alpha=0.85, label=label)

    ax.set_xlabel("iteration")
    ylabel = {
        "v_norms": r"$\|V\|^2$ / generator loss",
        "inner_losses": r"ICNN inner loss $\|\nabla\psi(x)-\bar y\|^2$",
        "target_v_norms": r"direct OT target drift $\|\bar y-x\|^2$",
        "fit_ratios": r"$\|V_{ICNN}\|^2 / \|\bar y-x\|^2$",
    }.get(metric, metric)
    ax.set_ylabel(ylabel)
    ax.set_title(f"ICNN ablation curves: {metric}")
    ax.grid(True, alpha=0.3)

    if min(len(runs), max_curves) <= 12:
        ax.legend(fontsize=8)

    fig.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()
    return fig, ax


### Run the architecture + Sinkhorn ablation

Start with the quick grid below. It is intentionally small so you can verify that the code runs.

Once it works, switch to the report-quality grid in the comments.

Interpretation:

- If larger tapered ICNNs reduce both `tail_mean_v_norm` and `tail_mean_inner_loss`, ICNN capacity was limiting.
- If `tail_mean_inner_loss` falls but `tail_mean_v_norm` does not, the generator update or target regularization may be limiting.
- If very small `sinkhorn_reg` produces unstable or noisy curves, use this to justify a moderate regularization value.
- If large `sinkhorn_reg` gives overly smooth targets, expect lower/noisier mode specificity.


In [ ]:
# Quick first pass: enough to check trends without spending too much compute.
# The config width=512, depth=4 corresponds to [512, 256, 128, 64].
ABLATION_WIDTHS = [128, 256, 512]
ABLATION_DEPTHS = [2, 3, 4]
ABLATION_SINKHORN_REGS = [0.02, 0.05, 0.1]
ABLATION_SEEDS = [42]
ABLATION_ITERS = 1000

# Report-quality version:
# ABLATION_WIDTHS = [128, 256, 512]
# ABLATION_DEPTHS = [2, 3, 4, 5]
# ABLATION_SINKHORN_REGS = [0.01, 0.02, 0.05, 0.1, 0.2]
# ABLATION_SEEDS = [0, 1, 2, 3, 4]
# ABLATION_ITERS = 3000

ablation_df, ablation_runs = run_icnn_architecture_sinkhorn_ablation(
    widths=ABLATION_WIDTHS,
    depths=ABLATION_DEPTHS,
    sinkhorn_regs=ABLATION_SINKHORN_REGS,
    seeds=ABLATION_SEEDS,
    target=TARGET,
    num_iters=ABLATION_ITERS,
    batch_size=BATCH_SIZE,
    inner_steps=INNER_STEPS,
    lr=LR,
    inner_lr=1e-2,
    strong_convexity=1.0,
    init_mode=ICNN_INIT_MODE,
    init_blend=ICNN_INIT_BLEND,
    device=DEVICE,
)

ablation_df


In [ ]:
ablation_summary = summarize_ablation(ablation_df)
ablation_summary


In [ ]:
for reg in ABLATION_SINKHORN_REGS:
    plot_architecture_heatmap(
        ablation_df,
        metric="tail_mean_v_norm",
        sinkhorn_reg=reg,
        save_path=f"results/icnn_architecture_sinkhorn_ablation/heatmap_tail_mean_v_norm_reg_{reg:g}.png",
    )

    plot_architecture_heatmap(
        ablation_df,
        metric="tail_mean_inner_loss",
        sinkhorn_reg=reg,
        save_path=f"results/icnn_architecture_sinkhorn_ablation/heatmap_tail_mean_inner_loss_reg_{reg:g}.png",
    )

plot_sinkhorn_reg_ablation(
    ablation_df,
    metric="tail_mean_v_norm",
    save_path="results/icnn_architecture_sinkhorn_ablation/sinkhorn_reg_tail_mean_v_norm.png",
)

plot_sinkhorn_reg_ablation(
    ablation_df,
    metric="tail_mean_inner_loss",
    save_path="results/icnn_architecture_sinkhorn_ablation/sinkhorn_reg_tail_mean_inner_loss.png",
)

plot_ablation_curves(
    ablation_runs,
    metric="v_norms",
    save_path="results/icnn_architecture_sinkhorn_ablation/curves_v_norms.png",
)

plot_ablation_curves(
    ablation_runs,
    metric="inner_losses",
    save_path="results/icnn_architecture_sinkhorn_ablation/curves_inner_losses.png",
)
